In [0]:

%sql
-- select * from dataplatform01_central_dev_catalog_01.bronze_sap_bo_snp.bo_user_role_classification
-------------CELL to create User profile table in Silver from bronze data---------------------
CREATE OR REPLACE TABLE dataplatform01_central_dev_catalog_01.silver_sap_bo.bo_user_profile
AS
SELECT
  cur.UserName,
  CASE
    WHEN cur.DisplayName = "Does not exist" THEN 
    CASE 
      WHEN o.DisplayName = "Does not exist" THEN "Unidentified"
      ELSE o.DisplayName
    END
    ELSE cur.DisplayName
  END AS DisplayName,
  CASE
    WHEN cur.Title = "Does not exist" THEN "Unidentified"
    WHEN cur.Title = "Not applicable" THEN "External Resource"
    ELSE cur.Title
  END AS Title,
  CASE 
    WHEN ur.SeniorLevel = "Manager" THEN "Team lead"
    ELSE "Individual Contributer"
  END AS SeniorLevel,
  CASE
    WHEN cur.BU = "Does not exist" THEN
      CASE
        WHEN o.BU IS NULL or o.BU = "Does not exist" THEN "Unidentified"
        ELSE "Left Organization"
        END
    ELSE cur.BU
  END AS BU,
  -- cur.Scope
  CASE 
    WHEN o.BU in ("Does not exist", "Unidentified") THEN "Unidentified"
    ELSE o.BU
  END AS BU_Old,
  CASE
    WHEN cur.EmailAddress = "Does not exist" THEN 
    CASE 
      WHEN o.EmailAddress = "Does not exist" THEN "Unidentified"
      ELSE o.EmailAddress
    END
    ELSE cur.EmailAddress
  END AS EmailAddress,
  -- cur.BUGroup,
  cur.Country,
  curdate() as ingestion_ts
FROM dataplatform01_central_dev_catalog_01.bronze_sap_bo_snp.bo_user_profile AS cur
LEFT JOIN (
  SELECT
    UserName,
    Business_Unit as BU,
    DisplayName,
    EmailAddress,
    ingestion_ts,
    ROW_NUMBER() OVER (
      PARTITION BY UserName
      ORDER BY ingestion_ts DESC
    ) AS rn
  FROM dataplatform01_central_dev_catalog_01.bronze_sap_bo_snp.bo_user_profile_old
) AS o
  ON cur.UserName = o.UserName
LEFT JOIN
dataplatform01_central_dev_catalog_01.bronze_sap_bo_snp.bo_user_role_classification as ur
on ur.Title=cur.Title
where o.rn = 1;
--------check the cases where user BU changed------
select * from dataplatform01_central_dev_catalog_01.silver_sap_bo.bo_user_profile
where BU != BU_Old



In [0]:
%sql
select count(distinct universe_id) from dataplatform01_central_dev_catalog_01.silver_sap_bo.bo_universe_profile;
CREATE OR REPLACE TABLE dataplatform01_central_dev_catalog_01.silver_sap_bo.bo_universe_profile
AS
select link.universe_id, link.universe_name, link.db_alias,
link.WEBI_Doc_ID as cms_webi_Doc_ID,
cms_webi.Report_CUID, 
cms_webi.Report_Name, cms_webi.report_instance,
curdate() as ingestion_ts 
from (Select distinct Universe_ID,
Universe_Name,
DB_Alias,
WEBI_Doc_ID
from dataplatform01_central_dev_catalog_01.bronze_sap_bo_snp.universe_webi_cms_link) as link
left join 
(select distinct Report_ID, Report_CUID, Report_Instance, Report_Name from dataplatform01_central_dev_catalog_01.bronze_sap_bo_snp.sapbo_full_doc_id
where Report_Type = "Webi") as cms_webi
on cms_webi.report_id=link.webi_doc_id;

%sql
select * from dataplatform01_central_dev_catalog_01.silver_sap_bo.bo_universe_profile
where Report_CUID is null

In [0]:
%sql
CREATE OR REPLACE TABLE dataplatform01_central_dev_catalog_01.silver_sap_bo.webi_user_access_audit
SELECT cms_webi.Report_ID as WEBI_Doc_ID, cms_webi.Report_Name as WEBI_Name, audit.Object as Audit_name, audit.Object_ID as Audit_id,
audit.Event_Type as event_type, audit.User as UserName, audit.Access_TS,  audit.Events as Events
from (select Object, Object_ID, Event_Type, User, make_timestamp(Year, Month, Day, Hour, 0, 0) as Access_TS, Events  from dataplatform01_central_dev_catalog_01.bronze_sap_bo_snp.bo_user_access_audit) as audit
LEFT join (select distinct Report_ID, Report_CUID, Report_Instance, Report_Name from dataplatform01_central_dev_catalog_01.bronze_sap_bo_snp.sapbo_full_doc_id
where Report_Type = "Webi") as cms_webi
on cms_webi.Report_CUID=audit.Object_ID;

%sql
select count(distinct Audit_id) from  dataplatform01_central_dev_catalog_01.silver_sap_bo.webi_user_access_audit;


In [0]:
%sql
-- describe table dataplatform01_central_dev_catalog_01.bronze_sap_bo_snp.cibt_wave2_analysis_arcade
CREATE OR REPLACE TABLE dataplatform01_central_dev_catalog_01.silver_sap_bo.cibt_wave2_arcade_list
select Table_Name, Impacted as Wave2_impacted, Used_in_MI, "2026-12-31" as Releas_date  from dataplatform01_central_dev_catalog_01.bronze_sap_bo_snp.cibt_wave2_analysis_arcade


In [0]:
%sql
SELECT DISTINCT lower(trim(Table_Name)) AS table_name
  FROM dataplatform01_central_dev_catalog_01.silver_sap_bo.cibt_wave2_arcade_list

WITH arcade AS (
  SELECT DISTINCT lower(trim(Table_Name)) AS table_name
  FROM dataplatform01_central_dev_catalog_01.silver_sap_bo.cibt_wave2_arcade_list
),
univ AS (
  SELECT Universe, Tables
  FROM dataplatform01_central_dev_catalog_01.bronze_sap_bo_snp.dev_sapbo_universe_tables
),
exploded AS (
  SELECT
    u.Universe,
    lower(trim(tbl)) AS table_name
  FROM univ u
  LATERAL VIEW explode(split(u.Tables, ',')) e AS tbl
),
matches AS (
  SELECT e.Universe, e.table_name
  FROM exploded e
  JOIN arcade a
    ON e.table_name = a.table_name
)
SELECT
  Universe,
  collect_set(table_name) AS matched_tables,
  CASE WHEN COUNT(*) >= 1 THEN 'Risk_Universe' ELSE 'Safe'  CASE WHEN COUNT(*) >= 1 THEN 'Risk_Universe' ELSE 'Safe' END AS risk_tag
FROM matches


In [0]:
%sql
CREATE OR REPLACE TABLE dataplatform01_central_dev_catalog_01.silver_sap_bo.webi_cms_file_metadata
select distinct cms.Report_ID,cms.Report_CUID,
cms.Report_Name,
cms.Report_Type,
cms.User_Login, cms.User_Full_Name,
w.WEBI_Doc_ID,
w.Folder_Id,
w.Full_path,
w.updated,
w.scheduled,
w.created as created_by,
w.lastAuthor as updated_by 
from 
dataplatform01_central_dev_catalog_01.bronze_sap_bo_snp.sapbo_full_doc_id cms
left join
(select distinct Document_Id as WEBI_Doc_ID, Document_CUID as WEBI_CUID, Document_name as WEBI_Name, Folder_Id, Full_path, updated, scheduled, created, lastAuthor from dataplatform01_central_dev_catalog_01.bronze_sap_bo_snp.webi_metadata_cms
union 
select distinct Document_Id as WEBI_Doc_ID, Document_CUID as WEBI_CUID, Document_name as WEBI_Name, Folder_Id, Full_path, updated, scheduled, created, lastAuthor from dataplatform01_central_dev_catalog_01.bronze_sap_bo_snp.webi_full_records_metadata) w
on cms.Report_ID=w.WEBI_Doc_ID
where cms.Report_Type="Webi"
and w.folder_id is not null
and cms.Report_Instance =false;



-- select count(*) from _sqldf;

In [0]:
%sql

Select distinct Document_Id as WEBI_ID, Document_CUID as WEBI_CUID,
Document_name as WEBI_Name,  
Folder_Id,
full_path as WEBI_Folder,
scheduled,
updated,
Number_of_Data_Providers,
Data_Provider_ID,
Data_Provider_Name,
DataSource_ID,
DataSource_CUID,
Data_Profider_Refresh_Time,
Connection_ID,
Connection_Name,
created as created_by,
lastAuthor as updated_by
from dataplatform01_central_dev_catalog_01.bronze_sap_bo_snp.webi_metadata_cms;



SELECT
  COUNT(DISTINCT CASE WHEN updated > Data_Profider_Refresh_Time THEN WEBI_ID END) AS count_updated_later,
  COUNT(DISTINCT CASE WHEN updated < Data_Profider_Refresh_Time THEN WEBI_ID END) AS count_updated_earlier,
  COUNT(DISTINCT CASE WHEN Data_Profider_Refresh_Time is null THEN WEBI_ID END) AS No_Data_provider_refresh_time,
  COUNT(DISTINCT CASE WHEN updated is null THEN WEBI_ID END) AS No_Update_time,
  COUNT(DISTINCT WEBI_ID) AS total
FROM _sqldf;

